# 231 — −1 / 0 / +1 Segmentation clustering

Builds blobs on the canonical sample set (via `s22_build_blob_feature_matrix`), then paints each
ERSP as a -1/0/+1 segmentation map (`s23_build_minus101_feature_matrix`), and clusters.
Outputs land in `outputs/clustering/{kmeans,hierarchical}/minus101/runs/<timestamp>/`.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from functions import lf_blob_clustering_config as cfg
from functions.lf_blob_metrics import s22_build_blob_feature_matrix
from functions.lf_minus101 import (
    s23_build_minus101_feature_matrix,
    q60_plot_minus101_overlays,
)
from functions import lf_cluster_run as R

SCRIPT_NAME = '231_minus101_clustering.ipynb'


## Config

In [ ]:
# ── data input ───────────────────────────────
INPUT_DIR = Path(r'\\\\nasac-m2.unige.ch\\m-HumanNeuronLab\\ANALYSIS\\FLM\\Analysis_LoraFanda\\01_FBM_Analysis\\outputs\\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

# ── blob segmentation (same params as 230 so blobs are identical) ─
VALLEY_PARAMS = cfg.VALLEY_PARAMS

# ── -101 downsampling ───────────────────────
SCALE = 1.0           # 100% — full ERSP resolution painted map. Set <1 to downsample.
SCORE_MIN = None      # set a float to gate per-blob; None keeps all

# ── clustering ───────────────────────────────
KMEANS_K_RANGE = [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
HC_METHOD      = 'average'
HC_METRIC      = 'euclidean'
RANDOM_STATE   = cfg.RANDOM_STATE

print('VALLEY_PARAMS:', VALLEY_PARAMS)
print('SCALE:', SCALE, '  KMEANS_K_RANGE:', KMEANS_K_RANGE)


## Load canonical dataset

Shared across 210/230/231/232 so cross-feature comparisons are valid.


In [ ]:
# Loads ERSPs from INPUT_DIR, drops non-neural channels, applies the
# high-activity gate. Single source of truth so all four clustering
# notebooks operate on the same sample set. Cached in
# 02_FBM_Clustering/outputs/_dataset/canonical/ for fast reload.
from functions.lf_dataset import prepare_dataset, DEFAULT_CACHE_DIR

df_meta, ersp_list, X_3d = prepare_dataset(INPUT_DIR, cache_dir=DEFAULT_CACHE_DIR)
print(f'\nCanonical dataset: {len(df_meta)} samples · X_3d.shape={X_3d.shape}')
df_meta.head()


## Build blobs on the canonical samples

Calls `s22_build_blob_feature_matrix` with the same `VALLEY_PARAMS` 230 uses — blobs are deterministic
and identical to 230's blobs. 231 doesn't depend on 230 having run first.


In [ ]:
_, _, blobs_per_sample = s22_build_blob_feature_matrix(
    ersp_list=ersp_list,
    max_blobs=int(VALLEY_PARAMS['max_blobs']),
    thr_pos=float(VALLEY_PARAMS['thr_pos']),
    thr_neg=float(VALLEY_PARAMS['thr_neg']),
    delta_valley=float(VALLEY_PARAMS['delta_valley']),
    min_mean_pos=float(VALLEY_PARAMS['min_mean_pos']),
    max_mean_neg=float(VALLEY_PARAMS['max_mean_neg']),
    sign_mode=str(VALLEY_PARAMS['sign_mode']),
)
print('blobs_per_sample built:', len(blobs_per_sample))


## Build -101 feature matrix

In [ ]:
X_101, ds_shape = s23_build_minus101_feature_matrix(
    ersp_list=ersp_list,
    blobs_per_sample=blobs_per_sample,
    scale=SCALE,
    score_min=SCORE_MIN,
)
print('X_101:', X_101.shape, '  ds_shape:', ds_shape)


## QC: painted -101 maps (sanity-check 30 random samples)

In [ ]:
rng = np.random.default_rng(42)
qc_idx = rng.choice(len(ersp_list), size=min(30, len(ersp_list)), replace=False).tolist()
q60_plot_minus101_overlays(
    indices=qc_idx,
    ersp_list=ersp_list,
    blobs_per_sample=blobs_per_sample,
    df_meta=df_meta,
    scale=SCALE,
    score_min=SCORE_MIN,
)


# Clustering

In [ ]:
manifest_km = R.fit_and_save(
    X_101,
    df_keep=df_meta,
    method='kmeans',
    feature_set='minus101',
    params={'k_range': KMEANS_K_RANGE, 'random_state': RANDOM_STATE, 'n_init': 20},
    method_label='K-Means',
    feature_set_label='\u22121 / 0 / +1 Segmentation',
    notebook=SCRIPT_NAME,
)
BEST_K = manifest_km['summary']['best_k']
print(f'Best K (KMeans/minus101, by silhouette): {BEST_K}')


In [ ]:
manifest_hc = R.fit_and_save(
    X_101,
    df_keep=df_meta,
    method='hierarchical',
    feature_set='minus101',
    params={'linkage': HC_METHOD, 'metric': HC_METRIC, 'k_range': KMEANS_K_RANGE},
    method_label='Hierarchical',
    feature_set_label='\u22121 / 0 / +1 Segmentation',
    notebook=SCRIPT_NAME,
)
print(f'Best K (HC/minus101, by silhouette): {manifest_hc["summary"]["best_k"]}')


## Per-cluster centroid PNGs (for the MOBA cluster chips)

Mean -1/0/+1 map across cluster members, reshaped to `ds_shape`.


In [ ]:
import json
CLUSTERING_DIR = Path(R.DEFAULT_OUTPUTS_ROOT)
INDEX_PATH = CLUSTERING_DIR / 'index.json'

def _save_per_cluster_centroid_pngs_minus101(manifest, X_local, ds_shape_local, *, vlim=1.0):
    if manifest['feature_set'] != 'minus101':
        return 0
    run_dir = CLUSTERING_DIR / manifest['method'] / manifest['feature_set'] / 'runs' / manifest['run_id']
    df = pd.read_csv(run_dir / 'labels.csv')
    cluster_col = f"cluster_{manifest['method']}_{manifest['feature_set']}"
    if cluster_col not in df.columns:
        cands = [c for c in df.columns if c.startswith('cluster_')]
        if not cands: return 0
        cluster_col = cands[0]
    labels = df[cluster_col].to_numpy()
    if len(labels) != X_local.shape[0]:
        print(f"  [skip] {manifest['run_id']}: labels ({len(labels)}) vs X_101 ({X_local.shape[0]}) mismatch")
        return 0
    out_dir = run_dir / 'cluster_centroids'
    out_dir.mkdir(parents=True, exist_ok=True)
    uniq = sorted(int(c) for c in np.unique(labels))
    for c in uniq:
        idx = np.where(labels == c)[0]
        mean_map = X_local[idx].mean(axis=0).reshape(ds_shape_local)
        fig, ax = plt.subplots(figsize=(2, 1.5))
        ax.imshow(mean_map, aspect='auto', origin='lower',
                  cmap='bwr', vmin=-vlim, vmax=vlim, interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        fig.savefig(out_dir / f'cluster_{int(c):02d}.png', dpi=80, bbox_inches='tight', pad_inches=0)
        plt.close(fig)
    return len(uniq)

if INDEX_PATH.exists():
    with open(INDEX_PATH) as f:
        idx_data = json.load(f)
    runs = [r for r in idx_data.get('runs', []) if r['feature_set'] == 'minus101']
    print(f'Backfilling for {len(runs)} minus101 runs...')
    for run in runs:
        mp = CLUSTERING_DIR / run['path'] / 'manifest.json'
        if not mp.exists(): continue
        manifest = json.loads(mp.read_text())
        n = _save_per_cluster_centroid_pngs_minus101(manifest, X_101, ds_shape)
        if n: print(f"  [{manifest['method']}/minus101] {manifest['run_id']} -> {n} PNGs")
    print('Done.')
